# Background comparison: ecoinvent 3.12 vs BAFU-2026

Compares the results of the same foreground models with 2 background databases.

1. Run `import_bafu.py` one time. It makes the project `Lasers4MaaS_BAFU` and imports BAFU-2026.
2. Run `joining_task`, `pre_post` and `workflow_assembly` two times: with `BACKGROUND = "ecoinvent"` and with `BACKGROUND = "bafu"`
   (or set the environment variable `L4M_BACKGROUND`). The results go to `results/` and `results/bafu/`.
3. Run this notebook.

A ratio above 1 means that BAFU gives a higher impact than ecoinvent.

In [ ]:
import bw2data as bd
import pandas as pd
import plotly.express as px
from IPython.display import display

from l4m_lca.compare import compare
from l4m_lca.constants import BACKGROUNDS
from l4m_lca.impacts import BackgroundImpacts
from l4m_lca.project import category, ef31_methods, find_activity, short

## Parameters

In [ ]:
LABELS = ("ecoinvent", "bafu")
SCENARIO = "amount"
RESULT_SETS = {
    "Joining": "State_of_the_Art_Dynamic_Beam_Shaping",
    "Pre-/post-joining": "Chemical_Pre_and_Post_processes_Mechanical_Pre_and_Post_processes",
    "Material (per kg)": "Material_production",
}

# Matching background inputs: label -> {background: (name, location)}
INPUTS = {
    "Electricity MV, RER": {"ecoinvent": ("market group for electricity, medium voltage", "RER"),
                            "bafu": ("Electricity, medium voltage, at grid", "RER")},
    "Electricity MV, NL": {"ecoinvent": ("market for electricity, medium voltage", "NL"),
                           "bafu": ("Electricity, medium voltage, at grid", "NL")},
    "Argon, liquid": {"ecoinvent": ("market for argon, liquid", "RER"), "bafu": ("Argon, liquid, at plant", "RER")},
    "Nitric acid": {"ecoinvent": ("market for nitric acid, without water, in 50% solution state", "RER w/o RU"),
                    "bafu": ("Nitric acid, 50% in H2O, at plant", "RER")},
    "Sodium hydroxide": {"ecoinvent": ("market for sodium hydroxide, without water, in 50% solution state", "RER"),
                         "bafu": ("Sodium hydroxide, 50% in H2O, production mix, at plant", "RER")},
    "Aluminium, wrought": {"ecoinvent": ("market for aluminium, wrought alloy", "GLO"),
                           "bafu": ("Aluminium, production mix, wrought alloy, at plant", "RER")},
    "Steel 18/8": {"ecoinvent": ("market for steel, chromium steel 18/8", "GLO"),
                   "bafu": ("Chromium steel 18/8, at plant", "RER")},
}

## Foreground results: BAFU / ecoinvent

In [ ]:
tables = []
for name, stem in RESULT_SETS.items():
    t = compare(BACKGROUNDS[LABELS[0]]["results"], BACKGROUNDS[LABELS[1]]["results"], stem, *LABELS, scenario=SCENARIO)
    tables.append(t.assign(set=name))
comparison = pd.concat(tables, ignore_index=True)
comparison["indicator"] = comparison["indicator"].map(short)
ratios = comparison.pivot_table(index=["set", "Process"], columns="indicator", values="ratio", sort=False)
display(ratios.style.format("{:.2f}").background_gradient(cmap="RdBu_r", vmin=0, vmax=2, axis=None))

In [ ]:
impact_ratios = comparison[~comparison["indicator"].str.contains(r"\(kWh\)|\(EUR\)")]
fig = px.strip(impact_ratios, x="ratio", y="indicator", color="set", hover_data=["Process"], log_x=True,
               title=f"BAFU / ecoinvent per process ({SCENARIO} scenario); 1 = same result")
fig.add_vline(x=1, line_dash="dash", line_color="#888")
fig.update_layout(template="plotly_white", height=520, width=1000)
display(fig)

## Background inputs: impact of 1 unit

This table explains the foreground differences: the foreground amounts are the same in both runs.

In [ ]:
unit_rows = []
for label in LABELS:
    profile = BACKGROUNDS[label]
    bd.projects.set_current(profile["project"])
    methods = ef31_methods()
    nodes = {k: find_activity(profile["database"], *v[label]) for k, v in INPUTS.items()}
    impacts = BackgroundImpacts(methods, next(iter(nodes.values())))
    for k, node in nodes.items():
        for m, value in zip(methods, impacts.unit(node)):
            unit_rows.append({"input": k, "indicator": short(category(m)), "background": label, "value": value})
units_df = pd.DataFrame(unit_rows).pivot_table(index=["input", "indicator"], columns="background", values="value", sort=False)
units_df["ratio"] = units_df[LABELS[1]] / units_df[LABELS[0]]
units_df["ratio"].unstack("indicator").style.format("{:.2f}").background_gradient(cmap="RdBu_r", vmin=0, vmax=2, axis=None)

## Import quality of BAFU-2026

Exchanges that did not link during the import are removed. Their emissions are missing from the BAFU results.

In [ ]:
unlinked = pd.read_csv(f"{BACKGROUNDS['bafu']['results']}/unlinked_exchanges.csv")
display(unlinked.groupby("type")["occurrences"].agg(["count", "sum"]))
unlinked.head(20)